In [2]:
# # Install Pytorch & other libraries
# %pip install "torch==2.4.1" tensorboard 
# %pip install flash-attn "setuptools<71.0.0" scikit-learn 
 
# # Install Hugging Face libraries
# %pip install  --upgrade \
#   "datasets==3.1.0" \
#   "accelerate==1.2.1" \
#   "hf-transfer==0.1.8"
#   #"transformers==4.47.1" \
 
# # ModernBERT is not yet available in an official release, so we need to install it from github
# %pip install "git+https://github.com/huggingface/transformers.git@6e0515e99c39444caae39472ee1b2fd76ece32f1" --upgrade

In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
from datasets import load_dataset, concatenate_datasets
 
# Dataset id from huggingface.co/dataset
dataset_id = "youralien/feedback_qesconv_16wayclassification"
dataset_id_care = "youralien/CARE_10percent_16wayclassification"

# Load raw dataset
raw_dataset = load_dataset(dataset_id, split="train") # happens to be called train
care_raw_dataset = load_dataset(dataset_id_care, split="train") # happens to be called train

print(f"FeedbackESConv Raw dataset size: {len(raw_dataset)}")
print(f"CARE raw dataset size: {len(care_raw_dataset)}")

/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FeedbackESConv Raw dataset size: 8179
CARE raw dataset size: 370


In [3]:
split_dataset = raw_dataset.train_test_split(test_size=0.05, seed=0)
print(f"Train dataset size: {len(split_dataset['train'])}")
print(f"Test dataset size: {len(split_dataset['test'])}")
split_dataset['train'][0]

Train dataset size: 7770
Test dataset size: 409


{'conv_index': 252,
 'helper_index': 9,
 'input': ["Seeker: I am sorry. I can't chat any more please end of this survey. Ok. I can try.",
  'Helper: I am sorry to hear that. I believe in you, things will be better with time. Would you like to continue chatting?',
  'Seeker: Yes',
  'Helper: Okay. Are you excited for the upcoming holidays?',
  'Seeker: Yeah, i am excited upcoming chrisms and new year party.',
  'Helper: That sounds fun! Is the new year party with friends? Or is it a family affair?',
  "Seeker: New year party is with my friend and family affairs. It's very excited and lot of fun and games.",
  'Helper: That is fun! Friends and family are the most wonderful cure for the blues. What types of games will you be playing?'],
 'Reflections-goodareas': 0,
 'Validation-goodareas': 0,
 'Empathy-goodareas': 1,
 'Questions-goodareas': 1,
 'Suggestions-goodareas': 0,
 'Self-disclosure-goodareas': 0,
 'Structure-goodareas': 0,
 'Professionalism-goodareas': 0,
 'Reflections-badareas': 

In [4]:
eval_set = concatenate_datasets([split_dataset['test'], care_raw_dataset])
eval_set

Dataset({
    features: ['conv_index', 'helper_index', 'input', 'Reflections-goodareas', 'Validation-goodareas', 'Empathy-goodareas', 'Questions-goodareas', 'Suggestions-goodareas', 'Self-disclosure-goodareas', 'Structure-goodareas', 'Professionalism-goodareas', 'Reflections-badareas', 'Validation-badareas', 'Empathy-badareas', 'Questions-badareas', 'Suggestions-badareas', 'Self-disclosure-badareas', 'Structure-badareas', 'Professionalism-badareas', 'therapist_id', 'chat_code', 'therapist_index', 'Session Management-goodareas', 'Session Management-badareas'],
    num_rows: 779
})

In [5]:
split_dataset['test'] = eval_set

In [6]:
print(f"Train dataset size: {len(split_dataset['train'])}")
print(f"Test dataset size: {len(split_dataset['test'])}")

Train dataset size: 7770
Test dataset size: 779


In [7]:
split_dataset['train'][0]['input'][-1]

'Helper: That is fun! Friends and family are the most wonderful cure for the blues. What types of games will you be playing?'

In [8]:
split_dataset['train'][0]['input'][-3:-1]

['Helper: That sounds fun! Is the new year party with friends? Or is it a family affair?',
 "Seeker: New year party is with my friend and family affairs. It's very excited and lot of fun and games."]

In [13]:
# Downsample once before training
majority_samples = split_dataset['train'].filter(lambda example: example['Suggestions-badareas'] == 0)
minority_samples = split_dataset['train'].filter(lambda example: example['Suggestions-badareas'] == 1)
print("Number of NOT SELECTED samples:", len(majority_samples))
print("Number of SELECTED samples:", len(minority_samples))
downsample_ratio = len(majority_samples)/len(minority_samples)
print("Downsample Ratio: ", downsample_ratio)
downsampled_majority = majority_samples.shuffle(seed=42).select(range(len(majority_samples) // int(downsample_ratio)))
balanced_dataset = concatenate_datasets([downsampled_majority, minority_samples]).shuffle(seed=42)

Number of NOT SELECTED samples: 6675
Number of SELECTED samples: 1095
Downsample Ratio:  6.095890410958904


In [14]:
balanced_dataset

Dataset({
    features: ['conv_index', 'helper_index', 'input', 'Reflections-goodareas', 'Validation-goodareas', 'Empathy-goodareas', 'Questions-goodareas', 'Suggestions-goodareas', 'Self-disclosure-goodareas', 'Structure-goodareas', 'Professionalism-goodareas', 'Reflections-badareas', 'Validation-badareas', 'Empathy-badareas', 'Questions-badareas', 'Suggestions-badareas', 'Self-disclosure-badareas', 'Structure-badareas', 'Professionalism-badareas'],
    num_rows: 2207
})

## Exploring Hyperparameter Sweeps with WanDB
Link: https://wandb.ai/matt24/vit-snacks-sweeps/reports/Hyperparameter-Search-for-HuggingFace-Transformer-Models--VmlldzoyMTUxNTg0

In [16]:
import wandb
wandb.login()


# %env WANDB_PROJECT=ModernBert_SkillClassifier
%env WANDB_PROJECT=Roberta_SkillClassifier
# false, checkpoint 
%env WANDB_LOG_MODEL=false

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: ryanlouie2021 (ryanlouie2021-stanford-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


env: WANDB_PROJECT=Roberta_SkillClassifier
env: WANDB_LOG_MODEL=false


### Targeted Sweep of Top Performing RoBERTa Hyperparams with Downsampling + Upweighting Majority Class

In [15]:
# https://wandb.ai/wandb_fc/articles/reports/What-Is-Bayesian-Hyperparameter-Optimization-With-Tutorial---Vmlldzo1NDQyNzcw
sweep_config = {
    'method': 'bayes',
    'metric': {
         'name': 'eval/f1', # important to use 'eval/f1' since this is the specific name
         'goal': 'maximize'  
    }
}

# hyperparameters
parameters_dict = {
    'epochs': {
        'values': [5, 10, 20] # this has a relationship with the linear learning rate schedule. Some folks tried 20!! Our experience is that many epochs is slow, and tends to overfit.
    },
    'batch_size': {
        'value': 16 # 128 wont fit into 24GB GPU memory
    },
    'warmup_ratio': {
        'values': [0.0, 0.1] # following 10% and then a linear decay; https://openreview.net/pdf?id=nzpLWnVAyah
    },
    'learning_rate': {
        'distribution': 'uniform',
        'min': 2e-6,
        'max': 8e-5, # 4.5e-6
    },
    # 'learning_rate': {
    #     'values': [7.4e-6, 9.8e-6]
    # },
    'weight_decay': {
        # 0.1, and 0.01 has been used in original RoBERTa GLUE, 0.2 was found to be actually good in some runs? 
        # 'values': [1e-6, 5e-6, 8e-6, 1e-5] # these smaller values were taken from the ModernBERT hyperparameter sweep of GLUE
        'values': [0.0, 0.06, 0.1, 0.2]
        # 'value': 0.0
    },
    # 'beta': {    
    #     'value': 0.999 # Since Beta ranges from [0, 1), we select several parameters that were found to be best in the original paper, also, selecting a few that don't use class balanced loss as much. 0.99 is the inverse loss.
    # },
    'context_size': {
        'values': [1, 5, None] # some skills, like Reflections, depend wholy on what was said previously, making it easier to learn. 10 was based on Rose's work. None is full available context
        # 'value': 1,
    },
    'downsampling_factor': {
        'values': [2, 4, 6]
    }
}

sweep_config['parameters'] = parameters_dict


In [17]:
import random
import torch
from torch.utils.data import Sampler, DataLoader

class ImbalancedDatasetSampler(Sampler):
    def __init__(self, dataset, indices=None, downsampling_factor=3):
        # Indices of all samples
        self.indices = list(range(len(dataset))) if indices is None else indices
        
        # Extract labels from the dataset
        self.labels = [dataset[i]['labels'] for i in self.indices]
        self.downsampling_factor = downsampling_factor
        
        # Store reference to the dataset
        self.dataset = dataset
        
    def __iter__(self):
        # Find indices of each class
        majority_indices = [i for i, label in zip(self.indices, self.labels) if label == 0]
        minority_indices = [i for i, label in zip(self.indices, self.labels) if label == 1]
        
        # Randomly select majority samples for this epoch
        random.shuffle(majority_indices)
        downsampled_majority = majority_indices[:len(majority_indices) // self.downsampling_factor]
        
        # Combine with all minority samples
        indices = downsampled_majority + minority_indices
        random.shuffle(indices)
        return iter(indices)
    
    def __len__(self):
        # The length is the number of samples that will be sampled
        labels = torch.tensor(self.labels)
        majority_count = (labels == 0).sum().item() // self.downsampling_factor
        minority_count = (labels == 1).sum().item()
        return majority_count + minority_count

def get_custom_dataloader(dataset, tokenizer, batch_size, downsampling_factor=3):
    # Create the sampler
    sampler = ImbalancedDatasetSampler(dataset, downsampling_factor=downsampling_factor)
    
    # Create data collator
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    # Create the dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
        collate_fn=data_collator
    )
    
    return dataloader

In [18]:
from transformers import AutoModelForSequenceClassification 
from transformers import DataCollatorWithPadding
from transformers import AutoTokenizer
from transformers import Trainer, TrainingArguments

from huggingface_hub import HfFolder

import torch
import gc

import evaluate
import numpy as np

def compute_metrics_fn(eval_preds):
    metrics = dict()
    
    accuracy_metric = evaluate.load('accuracy')
    precision_metric = evaluate.load('precision')
    recall_metric = evaluate.load('recall')
    f1_metric = evaluate.load('f1')
    
    logits = eval_preds.predictions
    labels = eval_preds.label_ids
    preds = np.argmax(logits, axis=-1)  
    
    metrics.update(accuracy_metric.compute(predictions=preds, references=labels))
    metrics.update(precision_metric.compute(predictions=preds, references=labels, average='binary'))
    metrics.update(recall_metric.compute(predictions=preds, references=labels, average='binary'))
    metrics.update(f1_metric.compute(predictions=preds, references=labels, average='binary'))

    # Print some predictions
    print(f"Some predictions: {preds[:10]}")
    
    return metrics


def get_class_weight(beta, n):
    """
    Compute class-balanced weight:
    alpha = (1 - beta) / (1 - beta^n) where n is the number of samples for the class
    
    Args:
        beta: Hyperparameter for class-balanced loss (typically between 0.9 and 0.999)
        n: Number of samples for a particular class
    
    Returns:
        The weight for the class
    """
    return (1 - beta) / (1 - beta**n)


def compute_class_balanced_loss(outputs, labels, num_items_in_batch, class_balanced_loss):
    """
    Compute class-balanced loss using the provided configuration
    
    Args:
        outputs: Model outputs containing 'logits'
        labels: Ground truth labels
        class_balanced_loss: Dictionary containing 'beta', 'n_0', and 'n_1' parameters
            - beta: Hyperparameter for class-balanced loss
            - n_0: Number of samples for class 0
            - n_1: Number of samples for class 1
    
    Returns:
        Computed loss value
    """
    logits = outputs['logits']
    
    # Compute class weights using the provided beta and class sample counts
    weights = torch.tensor([
        get_class_weight(class_balanced_loss['beta'], class_balanced_loss['n_0']),
        get_class_weight(class_balanced_loss['beta'], class_balanced_loss['n_1'])
    ])
    
    # Normalize weights
    weights = weights / weights.sum()
    
    # Move weights to the same device as logits
    weights = weights.to(device=logits.device)
    
    # Create loss function with computed weights
    criterion = torch.nn.CrossEntropyLoss(weight=weights)
    
    # Compute loss
    loss = criterion(logits, labels)
    
    return loss

def prepare_input_text(example, context_size=1):
    """
    [-6] Seeker: 
    [-5] Helper:
    [-4] Seeker: 
    [-3] Helper:
    [-2] Seeker: 
    [-1] Helper: Response to classify
    """
    # Convert the last two items of input list to a single text
    response_to_classify = example['input'][-1]
    if context_size is None:
        context = "\n".join(example['input'][:-1])
    else:
        context_start_idx = -1 - context_size
        context = "\n".join(example['input'][context_start_idx:-1])
    return {
        'text': f"{context}[SEP]{response_to_classify}",
        **{k:v for k,v in example.items() if k != 'input'}  # Keep other fields
    }

# def prepare_tokenized_binary_classification_dataset(dataset, which_class):
#     """
#     e.g., which_dataset = "Question-goodareas"
#     """
#     # Apply the preprocessing
#     dataset = dataset.map(prepare_input_text)
#     print(dataset['train'][0])
    
#     SKILL_OPTIONS = ["Reflections", "Validation", "Empathy", "Questions", "Suggestions", "Self-disclosure", "Structure", "Professionalism"]
#     goodareas_to_ignore = [f"{skill}-goodareas" for skill in SKILL_OPTIONS if f"{skill}-goodareas" != which_class]
#     badareas_to_ignore = [f"{skill}-badareas" for skill in SKILL_OPTIONS if f"{skill}-badareas" != which_class]
#     cols_to_remove = ['conv_index', 'helper_index', 'input', 'text']
#     cols_to_remove.extend(goodareas_to_ignore)
#     cols_to_remove.extend(badareas_to_ignore)
#     if which_class in dataset["train"].features.keys():
#         dataset =  dataset.rename_column(which_class, "labels") # to match Trainer
#     tokenized_dataset = dataset.map(lambda batch: tokenizer(batch['text'], truncation=True), batched=True, remove_columns=cols_to_remove)
#     return tokenized_dataset
    
def cleanup(things_to_delete: list | None = None):
    if things_to_delete is not None:
        for thing in things_to_delete:
            if thing is not None:
                del thing

    gc.collect()
    torch.cuda.empty_cache()

def train_model(config, dataset, which_class):

    # Model id to load the tokenizer
    # model_id, model_nickname = ("answerdotai/ModernBERT-large", "modernbert")
    model_id, model_nickname = ("FacebookAI/roberta-large", "roberta")
    
    # Load Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    with wandb.init(config=config):
        # set sweep configuration
        config = wandb.config

        def prepare_input_text_fn(example):
            return prepare_input_text(example, context_size=config.context_size)
        
        dataset = dataset.map(prepare_input_text_fn)
        print(dataset['train'][0])
        
        SKILL_OPTIONS = ["Reflections", "Validation", "Empathy", "Questions", "Suggestions", "Self-disclosure", "Structure", "Professionalism"]
        goodareas_to_ignore = [f"{skill}-goodareas" for skill in SKILL_OPTIONS if f"{skill}-goodareas" != which_class]
        badareas_to_ignore = [f"{skill}-badareas" for skill in SKILL_OPTIONS if f"{skill}-badareas" != which_class]
        cols_to_remove = ['conv_index', 'helper_index', 'input', 'text']
        cols_to_remove.extend(goodareas_to_ignore)
        cols_to_remove.extend(badareas_to_ignore)
        if which_class in dataset["train"].features.keys():
            dataset = dataset.rename_column(which_class, "labels") # to match Trainer
        tokenized_dataset = dataset.map(lambda batch: tokenizer(batch['text'], truncation=True), batched=True, remove_columns=cols_to_remove)
    
        # Prepare model labels - useful for inference
        labels = ["not selected", "selected"]
        num_labels = len(labels)
        label2id, id2label = dict(), dict()
        for i, label in enumerate(labels):
            label2id[label] = str(i)
            id2label[str(i)] = label
         
        # Download the model from huggingface.co/models
        model = AutoModelForSequenceClassification.from_pretrained(
            model_id, num_labels=num_labels, label2id=label2id, id2label=id2label
        )
        model.to('cuda')

        # Create custom dataloader for training
        train_dataloader = get_custom_dataloader(
            tokenized_dataset["train"], 
            tokenizer, 
            config.batch_size,
            downsampling_factor=config.downsampling_factor
        )

        # needs to be consistently named as current, because we'll be renaming this folder
        # OUTPUT_DIR = f"{model_nickname}-{which_class}-sweeps-current"
        OUTPUT_DIR = f'{model_nickname}-{which_class}-eval_FeedbackESConv5pp_CARE10pp-sweeps-current'
        
        # Define training args
        training_args = TrainingArguments(
            output_dir=OUTPUT_DIR,
            per_device_train_batch_size=config.batch_size,
            per_device_eval_batch_size=16,
            learning_rate=config.learning_rate,
            warmup_ratio=config.warmup_ratio, 
            num_train_epochs=config.epochs,
            weight_decay=config.weight_decay,
            bf16=True, # bfloat16 training 
            optim="adamw_torch_fused", # improved optimizer 
            # logging & evaluation strategies
            logging_strategy="epoch",
            logging_steps=100,
            eval_strategy="epoch",
            save_strategy="epoch", # epoch, no
            save_total_limit=2, # needs to be commented out if save_strategy=no
            metric_for_best_model="f1",
            load_best_model_at_end=True, # needs to be commented out if save_strategy=no
            # use_mps_device=True, # mps device is a mac thing
            # push to hub parameters
            report_to="wandb",
            push_to_hub=True,
            hub_strategy="every_save",
            hub_token=HfFolder.get_token(),
            use_legacy_prediction_loop=True,  # Important for custom dataloader
        )

        #####
        # OPTION 1: Returning to complete inverse function
        #####
        # class_distribution = dataset['train'].select_columns(['labels']).to_pandas().value_counts()
        # print("Class distribution:")
        # class_distribution = class_distribution / len(dataset['train'])
        # print(class_distribution)
        # inverse_weights = 1 / class_distribution
        # inverse_weights = inverse_weights.astype('float32')
        # inverse_weights.values

        # def compute_class_balanced_loss_fn(outputs, labels, num_items_in_batch):
        #     """depends on the class_distribution variable defined above"""
        #     logits = outputs['logits']
        #     criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor(inverse_weights.values, device=0))
        #     loss = criterion(logits, labels)
        #     return loss
        
        #####
        # OPTION 2: CBL 
        #####
        # def compute_class_balanced_loss_fn(outputs, labels, num_items_in_batch):
        #     return compute_class_balanced_loss(outputs, labels, num_items_in_batch, {
        #             'beta': config.beta,
        #             # 'beta': 0.99,
        #             'n_0': n_0,
        #             'n_1': n_1
        #         })

        ##########
        # Option 3: Downsample + Upweight Majority
        ##########
        def compute_class_balanced_loss_fn(outputs, labels, num_items_in_batch):
            logits = outputs['logits']
            
            # Define weights based on your downsampling factor
            # If you downsampled by factor of 3, the weight for majority class should be 3
            weights = torch.tensor([config.downsampling_factor, 1.0], device=logits.device)  # [majority_weight, minority_weight]
            
            criterion = torch.nn.CrossEntropyLoss(weight=weights)
            loss = criterion(logits, labels)
            return loss
        
        hf_data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
        
        
        # Create a Trainer instance
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_dataset["train"],
            eval_dataset=tokenized_dataset["test"],
            processing_class=tokenizer,
            data_collator=hf_data_collator,
            compute_metrics=compute_metrics_fn,
            compute_loss_func=compute_class_balanced_loss_fn, # custom class balanced loss
        )
        # Override the default dataloader
        trainer.get_train_dataloader = lambda: train_dataloader
        try:
            trainer.train()
            cleanup(things_to_delete=[tokenized_dataset, hf_data_collator])
            return model, trainer, tokenizer
            # cleanup(things_to_delete=[model, trainer, tokenizer, tokenized_dataset, hf_data_collator])
        except:
            cleanup(things_to_delete=[model, trainer, tokenizer, tokenized_dataset, hf_data_collator])


In [19]:
import ipdb
from transformers.modelcard import parse_log_history
import shutil
import time

def run_sweep(which_class):
    WANDB_TEAM = "ryanlouie2021-stanford-university"
    WANDB_PROJECT = f'roberta-{which_class}-eval_FeedbackESConv5pp_CARE10pp-sweeps'
    # WANDB_PROJECT = f'modernbert-{which_class}-sweeps'
    sweep_id = wandb.sweep(sweep_config, project=WANDB_PROJECT)
    wandb_api = wandb.Api()
    def config_fn(config=None):
        model, trainer, tokenizer = train_model(config=config, dataset=split_dataset, which_class=which_class)
        train_log, eval_lines, eval_results = parse_log_history(trainer.state.log_history)
        current_f1scores = [line['F1'] for line in eval_lines]
        current_max_f1score = max(current_f1scores)
        print("Current Run Max F1 score: ", current_max_f1score)
        
        # now query wandb for most up-to-date sweep results
        sweep = wandb_api.from_path(f'{WANDB_TEAM}/{WANDB_PROJECT}/sweeps/{sweep_id}')
        # best_run = sweep.best_run() # problem with this is determines best run based on the final f1, not an intermediate checkpoint
        # best_history = best_run.scan_history(keys=["eval/f1"])
        # best_f1scores = [row["eval/f1"] for row in best_history]
        def max_f1score_from_run_history(run):
            history = run.scan_history(keys=["eval/f1"])
            f1scores = [row["eval/f1"] for row in history]
            return max(f1scores)
        runs_max_f1scores = [max_f1score_from_run_history(run) for run in sweep.runs]
        best_max_f1score = max(runs_max_f1scores)
        
        print("Best Run max F1 scores", best_max_f1score)
        
        # if the current is the best
        if current_max_f1score >= best_max_f1score:
            print("Found a new best model. Storing this new best model")
            # optionally push the best to hub now
            trainer.create_model_card()
            trainer.push_to_hub()
            
            # the checkpoints are already saved, but just organizing folder to be named best repo
            timestamp = int(time.time())
            shutil.move(f"{WANDB_PROJECT}-current", f"{WANDB_PROJECT}-best-{sweep_id}-{timestamp}")

        cleanup(things_to_delete=[model, trainer, tokenizer])
    
    wandb.agent(sweep_id, config_fn, count=64)

# classifier_types = ['goodareas', 'badareas']
classifier_types = ['badareas']
# SKILL_OPTIONS = ["Reflections", "Validation", "Empathy", "Questions", "Suggestions", "Self-disclosure", "Structure", "Professionalism"]
SKILL_OPTIONS = ["Suggestions"]
for classifier_type in classifier_types:
    for skill in SKILL_OPTIONS:
        run_sweep(f"{skill}-{classifier_type}")

Create sweep with ID: qpkjg3iw
Sweep URL: https://wandb.ai/ryanlouie2021-stanford-university/roberta-Suggestions-badareas-eval_FeedbackESConv5pp_CARE10pp-sweeps/sweeps/qpkjg3iw


wandb: Agent Starting Run: s2g4stkw with config:
wandb: 	batch_size: 16
wandb: 	context_size: 5
wandb: 	downsampling_factor: 2
wandb: 	epochs: 10
wandb: 	learning_rate: 2.1663781604632168e-05
wandb: 	warmup_ratio: 0
wandb: 	weight_decay: 0
ERROR:wandb.jupyter:Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 779/779 [00:00<00:00, 3915.11 examples/s]


{'conv_index': 252, 'helper_index': 9, 'input': ["Seeker: I am sorry. I can't chat any more please end of this survey. Ok. I can try.", 'Helper: I am sorry to hear that. I believe in you, things will be better with time. Would you like to continue chatting?', 'Seeker: Yes', 'Helper: Okay. Are you excited for the upcoming holidays?', 'Seeker: Yeah, i am excited upcoming chrisms and new year party.', 'Helper: That sounds fun! Is the new year party with friends? Or is it a family affair?', "Seeker: New year party is with my friend and family affairs. It's very excited and lot of fun and games.", 'Helper: That is fun! Friends and family are the most wonderful cure for the blues. What types of games will you be playing?'], 'Reflections-goodareas': 0, 'Validation-goodareas': 0, 'Empathy-goodareas': 1, 'Questions-goodareas': 1, 'Suggestions-goodareas': 0, 'Self-disclosure-goodareas': 0, 'Structure-goodareas': 0, 'Professionalism-goodareas': 0, 'Reflections-badareas': 0, 'Validation-badareas':

Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 779/779 [00:00<00:00, 3366.87 examples/s]
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: WARNING Config item 'learning_rate' was locked by 'sweep' (ignored update).
wandb: WARNING Config item 'weight_decay' was locked by 'sweep' (ignored

...

/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/trainer_pt_utils.py:478: FutureWarning: DistributedTensorGatherer is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


Some predictions: [0 0 0 0 0 0 0 0 0 1]


/nlp/scr/rylouie/miniconda3/envs/llama/lib/python3.12/site-packages/transformers/trainer_pt_utils.py:478: FutureWarning: DistributedTensorGatherer is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


Some predictions: [0 0 0 0 0 0 0 0 1 0]


IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [27]:
wandb_api = wandb.Api()
sweep = wandb_api.from_path('ryanlouie2021-stanford-university/roberta-Empathy-goodareas-sweeps/sweeps/z6e5n18d')
print(sweep)

<Sweep ryanlouie2021-stanford-university/roberta-Empathy-goodareas-sweeps/z6e5n18d (RUNNING)>


In [36]:
best_run = sweep.best_run()
history = best_run.scan_history(keys=["eval/f1"])
f1scores = [row["eval/f1"] for row in history]
f1scores

wandb: Sorting runs by -summary_metrics.eval/f1


[0.7289473684210527, 0.7368421052631579, 0.746922024623803, 0.7523680649526387]

In [51]:
last_run = sweep.runs[3]
history = last_run.scan_history(keys=["eval/f1"])
f1scores = [row["eval/f1"] for row in history]
f1scores

[0.7141041931385006,
 0.7409326424870466,
 0.7391304347826086,
 0.7503410641200545]

## Using the model to make predictions

In [41]:
import pandas as pd

# condition = "control"
# condition = "treatment"
# input_data = pd.read_csv(f"./all_{condition}_seekerhelper_pairs.csv")
input_data = pd.read_csv("N94_all_seekerhelper_pairs.csv")
print(len(input_data))
input_data.head()

3842


,id,seeker_post,response_post
0,14_a1df3c7155d0438b9c4084b57b66c6e6_0_0,NaN,good evening I understand you're feeling isola...
1,14_a1df3c7155d0438b9c4084b57b66c6e6_0_1,"Yeah, it's just... everyone was with their fam...",I hear you. It can be difficult growing apart ...
2,14_a1df3c7155d0438b9c4084b57b66c6e6_0_2,"Yeah, it's tough. You know, during the holiday...",Perhaps they're feeling similarly. Waiting for...
3,14_a1df3c7155d0438b9c4084b57b66c6e6_0_3,But why should I always have to be the bigger ...,"Being the bigger person can feel burdensome, c..."
4,14_a1df3c7155d0438b9c4084b57b66c6e6_0_4,"Yeah, exactly. It's like, why should I keep pu...",How would you like for them to show you they c...


In [42]:
from datasets import Dataset

study_dataset = Dataset.from_pandas(input_data)

wandb: ERROR Problem finishing run


In [43]:
from transformers import pipeline

# WHICH_CLASS="Reflections-goodareas"
WHICH_CLASS="Questions-goodareas"
# load model from huggingface.co/models using our repository id
# classifier = pipeline("sentiment-analysis", model=f"./roberta-Reflections-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-d6x1jzik-1741277930", device=0)
classifier = pipeline("sentiment-analysis", model=f"./roberta-Questions-goodareas-eval_FeedbackESConv5pp_CARE10pp-sweeps-best-82jc07j0-1741329550", device=0)

# classifier = pipeline("sentiment-analysis", model="ModernBERT-Empathy-goodareas-classifier", device=0)

# sample = f"Seeker: {input_data.loc[0, "seeker_post"]}\nHelper: {input_data.loc[0, "response_post"]}"
# pred = classifier(sample)
# print(pred)

def binary_prediction_seeker_response_post(seeker, helper):
    # sample = f"Seeker: {seeker}\nHelper: {helper}"
    sample = f"Seeker: {seeker}[SEP]Helper: {helper}"
    
    pred = classifier(sample)
    return int(pred[0]['label'] == 'selected')

Device set to use cuda:0


In [44]:
# output_preds = input_data.apply(binary_prediction_seeker_response_post, axis=0)

def predict_reflection(example):
    # Apply your binary prediction function to each example
    example["prediction"] = binary_prediction_seeker_response_post(
        example["seeker_post"], 
        example["response_post"]
    )
    return example

# Apply the function to the entire dataset at once
predicted_dataset = study_dataset.map(predict_reflection)

# strengths = [binary_prediction_seeker_response_post(input_data.loc[i, "seeker_post"], input_data.loc[i, "response_post"])
#              for i in range(len(input_data))]

Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3842/3842 [00:38<00:00, 99.94 examples/s]


In [45]:
input_data[f"{WHICH_CLASS}"] = predicted_dataset['prediction']
# input_data[f"{WHICH_CLASS}"] = strengths
# input_data["Empathy-goodareas"] = strengths

In [46]:
print(len(input_data))
input_data.head()

3842


,id,seeker_post,response_post,Questions-goodareas
0,14_a1df3c7155d0438b9c4084b57b66c6e6_0_0,NaN,good evening I understand you're feeling isola...,0
1,14_a1df3c7155d0438b9c4084b57b66c6e6_0_1,"Yeah, it's just... everyone was with their fam...",I hear you. It can be difficult growing apart ...,0
2,14_a1df3c7155d0438b9c4084b57b66c6e6_0_2,"Yeah, it's tough. You know, during the holiday...",Perhaps they're feeling similarly. Waiting for...,0
3,14_a1df3c7155d0438b9c4084b57b66c6e6_0_3,But why should I always have to be the bigger ...,"Being the bigger person can feel burdensome, c...",0
4,14_a1df3c7155d0438b9c4084b57b66c6e6_0_4,"Yeah, exactly. It's like, why should I keep pu...",How would you like for them to show you they c...,1


In [47]:
# input_data.to_csv(f'all_{condition}_seekerhelper_pairs_{WHICH_CLASS}.csv')
input_data.to_csv(f"N94_all_seekerhelper_pairs_{WHICH_CLASS}.csv")

In [37]:
f'all_{condition}_seekerhelper_pairs_{WHICH_CLASS}.csv'

'all_control_seekerhelper_pairs_Reflections-goodareas.csv'